## Imports and Dataset

In [ ]:
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
RUN_ID = "6f701059-94c7-4969-8ca9-5ba2656995d9"
data_path = Path("../data/interim") / RUN_ID / f"validated_dataset_{RUN_ID}.parquet"

df = pd.read_parquet(data_path)

## Target Column Analysis

In [ ]:
df.loan_status.value_counts()

In [ ]:
df.loan_status.value_counts(normalize=True).sort_values(ascending=False)

In [ ]:
status_counts = df["loan_status"].value_counts()

plt.figure(figsize=(10, 6))

sns.barplot(
    x=status_counts.values,
    y=status_counts.index
)

plt.title("Loan Status Distribution")
plt.xlabel("Count")
plt.ylabel("Loan Status")

plt.show()

#### Row Filtering

In [ ]:
good_loans = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

bad_loans = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

excluded_loans = [
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)"
]

valid_status = good_loans + bad_loans

In [ ]:
df_target = df[df["loan_status"].isin(valid_status)].copy()

print("Original dataset:", df.shape)
print("Filtered dataset:", df_target.shape)

#### Target Encoding

In [ ]:
target_map = {status: 0 for status in good_loans}
target_map.update({status: 1 for status in bad_loans})

df_target["target"] = df_target["loan_status"].map(target_map)

df_target[["loan_status", "target"]].sample()

#### EDA & Visualization

In [ ]:
print("Target distribution:")

df_target["target"].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df_target, x="target")

plt.title("Loan Default Distribution")
plt.xlabel("Target")
plt.ylabel("Count")

plt.show()

In [ ]:
# Default rate per year
df_target["issue_d"] = pd.to_datetime(df_target["issue_d"])

default_rate_per_year = (
    df_target.groupby(df_target["issue_d"].dt.year)["target"].mean()
)

default_rate_per_year

In [ ]:

plt.figure(figsize=(10, 6))

default_rate_per_year.plot(marker="o")

plt.title("Default Rate Per Year")
plt.xlabel("Year")
plt.ylabel("Default Rate")

plt.grid(True)
plt.show()

In [ ]:
df_target.groupby("loan_status")["target"].value_counts()

## Final Dataset

In [ ]:
print("Final dataset shape:", df_target.shape)

df_target.head()

In [ ]:
df_target["target"].isna().sum()

In [ ]:
df_target["target"].value_counts(normalize=True)